# 레슨 03 — pandas Series와 DataFrame 기초 정답지

> 교사·관리자 전용. 학생에게 배포하지 않는다.

이 정답지는 학생용 `mission.md` 의 문제 1~15와 번호가 1:1로 대응한다. pandas 첫 수업이므로 모범 답안과 줄 단위로 같을 필요는 없지만, DataFrame 구조 이해, 조건 필터, 계산 열, 집계 결과가 맞아야 한다.

## 환경 셀

In [ ]:
import os
import pandas as pd

IS_COLAB = "COLAB_GPU" in os.environ or "COLAB_TPU_ADDR" in os.environ
if IS_COLAB:
    DATA_BASE = "https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-data-analysis/lectures/03/data"
else:
    DATA_BASE = "./data"

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print("pandas:", pd.__version__)
print("data base:", DATA_BASE)

---

## 문제 1 정답 — 주문 데이터 불러오기

In [ ]:
df = pd.read_csv(f"{DATA_BASE}/sales_orders.csv")

print("shape:", df.shape)
print("columns:", list(df.columns))
print("앞 8행:")
print(df.head(8))
print("뒤 3행:")
print(df.tail(3))

rows, cols = df.shape
print(f"이 데이터는 {rows:,}행 {cols}열 구조입니다.")

### 왜 이 코드가 정답인지

`pd.read_csv` 는 CSV를 표 형태의 `DataFrame` 으로 읽는다. `shape` 는 행과 열 개수를 보여주므로 데이터가 기대한 1,000행 9열로 들어왔는지 바로 확인할 수 있다. `columns` 는 열 이름을 확인하는 가장 빠른 방법이다. `head()` 와 `tail()` 을 함께 보면 앞부분만 보고 데이터 전체 구조를 오해하는 일을 줄일 수 있다.

**채점 기준**

| 항목 | 통과 기준 |
|---|---|
| 파일 로드 | `pd.read_csv` 로 `df` 생성 |
| 구조 확인 | `shape`, `columns`, 앞뒤 일부 확인 |
| 해석 | 1,000행 9열이라고 말할 수 있음 |

---

## 문제 2 정답 — dtypes와 info 확인

In [ ]:
print("dtypes:")
print(df.dtypes)

print("info:")
df.info()

missing = df.isna().sum()
print("열별 결측 개수:")
print(missing)

print("숫자형 요약:")
print(df.describe())

print("결측치 총합:", missing.sum())
print("결측치 없음:", missing.sum() == 0)

### 왜 이 코드가 정답인지

분석 전에 열의 자료형을 확인해야 수치 계산이 가능한 열과 범주형 열을 구분할 수 있다. `dtypes` 는 열별 자료형을 보여주고, `info()` 는 행 개수와 null 여부를 함께 보여준다. `isna().sum()` 은 결측 개수를 숫자로 확인하는 방법이다. `describe()` 는 기본적으로 숫자형 열의 평균, 표준편차, 사분위수 등을 요약하므로 주문 수량, 단가, 할인율의 범위를 빠르게 볼 수 있다.

**지도 메모**

학생이 `info()` 출력 뒤에 `None` 이 같이 보이더라도 개념상 큰 문제는 아니다. 다만 `print(df.info())` 보다 `df.info()` 만 실행하는 편이 깔끔하다고 안내한다.

---

## 문제 3 정답 — Series와 DataFrame 구분

In [ ]:
qty = df["quantity"]
selected = df[["order_id", "category", "quantity", "unit_price"]]

print("qty type:", type(qty))
print(qty.head())
print("selected type:", type(selected))
print(selected.head())

print("qty shape:", qty.shape)
print("selected shape:", selected.shape)

### 왜 이 코드가 정답인지

열 하나를 문자열로 선택하면 `Series` 가 된다. 여러 열을 리스트로 선택하면 `DataFrame` 이 된다. `df["quantity"]` 와 `df[["quantity"]]` 는 비슷해 보이지만 결과 타입이 다르다. 이후 `groupby`, 정렬, 열 추가를 할 때 Series와 DataFrame 차이를 알고 있어야 코드 결과를 예측할 수 있다.

**자주 틀리는 답안**

| 답안 | 문제 |
|---|---|
| `df["order_id", "category"]` | pandas는 여러 열을 튜플이 아니라 리스트로 받음 |
| `df[["quantity"]]` 를 Series라고 말함 | 대괄호 리스트라 DataFrame |
| `df.quantity` 만 사용 | 가능은 하지만 열 이름에 공백/특수문자가 있으면 취약 |

---

## 문제 4 정답 — 수량과 단가 기본 통계

In [ ]:
quantity_mean = df["quantity"].mean()
quantity_median = df["quantity"].median()
quantity_max = df["quantity"].max()
quantity_min = df["quantity"].min()

price_mean = df["unit_price"].mean()
price_median = df["unit_price"].median()
price_max = df["unit_price"].max()
price_min = df["unit_price"].min()

print(f"수량 평균/중앙/최대/최소: {quantity_mean:.2f}, {quantity_median:.0f}, {quantity_max}, {quantity_min}")
print(f"단가 평균/중앙/최대/최소: {price_mean:,.0f}, {price_median:,.0f}, {price_max:,.0f}, {price_min:,.0f}")

large_quantity_count = (df["quantity"] >= 4).sum()
above_mean_price_count = (df["unit_price"] > price_mean).sum()

print("수량 4개 이상 주문 수:", large_quantity_count)
print("평균 단가보다 비싼 주문 수:", above_mean_price_count)
print("평균-중앙 단가 차이:", f"{price_mean - price_median:,.0f}원")

### 왜 이 코드가 정답인지

`quantity` 와 `unit_price` 는 숫자형 열이라 평균, 중앙값, 최댓값, 최솟값을 직접 계산할 수 있다. 조건식 `(df["quantity"] >= 4)` 는 True/False Series이고, True는 1처럼 합산되므로 `.sum()` 으로 주문 수를 셀 수 있다. 평균 단가가 중앙 단가보다 크면 고가 상품이 평균을 위로 끌어올렸을 가능성이 있다.

**예상 핵심값**

| 항목 | 값 |
|---|---:|
| 수량 평균 | 약 1.91 |
| 수량 최댓값 | 5 |
| 수량 4개 이상 주문 수 | 108 |

---

## 문제 5 정답 — 범주형 코드 고유값 확인

In [ ]:
code_columns = ["region", "channel", "category", "customer_grade"]

for col in code_columns:
    print(col, "고유값:", sorted(df[col].unique()))
    print(col, "고유값 개수:", df[col].nunique())

channel_counts = df["channel"].value_counts()
category_counts = df["category"].value_counts()

print("채널 주문 수:")
print(channel_counts)
print("카테고리 주문 수:")
print(category_counts)

print("주문 수 1위 채널:", channel_counts.idxmax())
print("주문 수 1위 카테고리:", category_counts.idxmax())

### 왜 이 코드가 정답인지

`unique()` 는 실제로 등장한 코드값을 보여주고, `nunique()` 는 종류 수를 숫자로 알려준다. `value_counts()` 는 범주형 데이터의 기본 요약 도구다. 평균을 낼 수 없는 문자열 열도 개수를 세면 분포를 파악할 수 있다. `idxmax()` 는 개수가 가장 큰 라벨을 반환하므로 주문 수 1위 채널과 카테고리를 바로 찾을 수 있다.

**예상 핵심값**

```text
채널 주문 수: W 432, A 380, S 188
카테고리 주문 수: FD 227, BK 219, ST 207, GD 188, TY 159
```

---

## 문제 6 정답 — loc와 iloc로 행 선택하기

In [ ]:
loc_rows = df.loc[:5]
iloc_rows = df.iloc[:5]

print("df.loc[:5] 행 수:", len(loc_rows))
print(loc_rows)

print("df.iloc[:5] 행 수:", len(iloc_rows))
print(iloc_rows)

print("loc 컬럼 선택:")
print(df.loc[10:15, ["order_id", "channel", "category"]])

print("iloc 컬럼 선택:")
print(df.iloc[10:15, [0, 3, 4]])

# loc는 라벨 기반이라 끝 라벨 5를 포함하고, iloc는 위치 기반이라 끝 위치 5를 포함하지 않는다.

### 왜 이 코드가 정답인지

`.loc` 는 인덱스 라벨을 기준으로 선택하고, 슬라이스 끝 라벨을 포함한다. 기본 인덱스가 0부터 시작할 때 `df.loc[:5]` 는 라벨 0, 1, 2, 3, 4, 5를 포함해 6행이다. `.iloc` 는 위치를 기준으로 선택하며 Python 슬라이싱처럼 끝 위치를 제외한다. 그래서 `df.iloc[:5]` 는 위치 0~4까지 5행이다.

**채점 포인트**

- `.loc` 와 `.iloc` 의 차이를 설명했는가.
- 열 이름 선택과 열 위치 선택을 각각 맞게 사용했는가.

---

## 문제 7 정답 — 단일 조건 필터

In [ ]:
app_orders = df[df["channel"] == "A"]
gadget_orders = df[df["category"] == "GD"]
discount_orders = df[df["discount_rate"] >= 0.10]

print("앱 채널 주문 수:", len(app_orders))
print(app_orders.head(3))

print("기기 카테고리 주문 수:", len(gadget_orders))
print(gadget_orders.head(3))

print("10% 이상 할인 주문 수:", len(discount_orders))
print(discount_orders.head(3))

### 왜 이 코드가 정답인지

`df["channel"] == "A"` 는 각 행이 앱 채널인지 판단하는 Boolean Series다. 이 Series를 `df[...]` 에 넣으면 True인 행만 남는다. 할인율 조건은 10% 이상이므로 `>= 0.10` 을 사용해야 한다. `> 0.10` 으로 쓰면 정확히 10% 할인 주문이 빠져 요구사항과 달라진다.

**예상 핵심값**

| 필터 | 행 수 |
|---|---:|
| 앱 채널 | 380 |
| 기기 카테고리 | 188 |
| 10% 이상 할인 | 292 |

---

## 문제 8 정답 — 복합 조건 필터

In [ ]:
vip_gadget = df[(df["customer_grade"] == "V") & (df["category"] == "GD")]
app_food = df[(df["channel"] == "A") & (df["category"] == "FD")]
bulk_discount = df[(df["quantity"] >= 4) & (df["discount_rate"] > 0)]

print("VIP 기기 주문 수:", len(vip_gadget))
print("앱 식품 주문 수:", len(app_food))
print("수량 4개 이상 + 할인 주문 수:", len(bulk_discount))
print(vip_gadget.head(5))

### 왜 이 코드가 정답인지

pandas 조건은 Series 단위 연산이다. Python의 `and` 는 Series 전체를 하나의 참/거짓으로 판단하려 하므로 사용할 수 없다. 대신 `&` 를 사용하고, 연산 우선순위 문제를 피하기 위해 각 조건을 괄호로 감싼다. 이 문제는 복합 조건을 안전하게 만드는 문법을 확인하는 핵심 문제다.

**예상 핵심값**

```text
VIP 기기 주문 수: 25
앱 식품 주문 수: 88
```

---

## 문제 9 정답 — 계산 열 만들기

In [ ]:
df["gross_revenue"] = df["quantity"] * df["unit_price"]
df["discount_amount"] = df["gross_revenue"] * df["discount_rate"]
df["net_revenue"] = df["gross_revenue"] - df["discount_amount"]

print(df[["quantity", "unit_price", "discount_rate", "gross_revenue", "discount_amount", "net_revenue"]].head())

total_net = df["net_revenue"].sum()
mean_net = df["net_revenue"].mean()
median_net = df["net_revenue"].median()

print("총 순매출:", f"{total_net:,.0f}원")
print("평균 주문금액:", f"{mean_net:,.0f}원")
print("중앙 주문금액:", f"{median_net:,.0f}원")

### 왜 이 코드가 정답인지

정가 매출은 수량과 단가를 곱한 값이다. 할인 금액은 정가 매출에 할인율을 곱한 값이고, 순매출은 정가 매출에서 할인 금액을 뺀 값이다. `gross_revenue * (1 - discount_rate)` 로 한 줄에 계산해도 되지만, 초반 수업에서는 `discount_amount` 를 따로 만들면 할인 금액과 순매출의 차이를 더 명확히 볼 수 있다.

**예상 핵심값**

| 항목 | 값 |
|---|---:|
| 총 순매출 | 37,546,040원 |
| 평균 주문금액 | 37,546원 |
| 중앙 주문금액 | 28,050원 |

---

## 문제 10 정답 — 순매출 상위 주문 찾기

In [ ]:
top10 = df.sort_values("net_revenue", ascending=False).head(10)
bottom10 = df.sort_values("net_revenue", ascending=True).head(10)

cols = ["order_id", "category", "quantity", "unit_price", "discount_rate", "net_revenue"]

print("순매출 상위 10개:")
print(top10[cols])

top_order = top10.iloc[0]
print("상위 1위 주문 ID:", int(top_order["order_id"]))
print("상위 1위 카테고리:", top_order["category"])
print("상위 1위 순매출:", f"{top_order['net_revenue']:,.0f}원")

print("하위 10개 평균 순매출:", f"{bottom10['net_revenue'].mean():,.0f}원")

### 왜 이 코드가 정답인지

`sort_values("net_revenue", ascending=False)` 는 순매출이 큰 주문부터 정렬한다. `.head(10)` 을 붙이면 상위 10개만 볼 수 있다. 정렬 결과에서 첫 행은 순매출 1위 주문이다. `iloc[0]` 은 정렬된 표의 첫 번째 위치를 선택하므로 원래 인덱스가 무엇이든 안정적으로 1위 행을 꺼낼 수 있다.

**예상 핵심값**

```text
상위 1위 주문 ID: 219
상위 1위 카테고리: GD
상위 1위 순매출: 245,000원
```

---

## 문제 11 정답 — 카테고리별 매출 집계

In [ ]:
category_order_count = df.groupby("category")["order_id"].count().sort_values(ascending=False)
category_net_sum = df.groupby("category")["net_revenue"].sum().sort_values(ascending=False)
category_net_mean = df.groupby("category")["net_revenue"].mean().sort_values(ascending=False)

print("카테고리별 주문 수:")
print(category_order_count)

print("카테고리별 순매출 합계:")
print(category_net_sum)

print("카테고리별 평균 주문금액:")
print(category_net_mean)

print("순매출 합계 1위 카테고리:", category_net_sum.idxmax())
print("평균 주문금액 1위 카테고리:", category_net_mean.idxmax())

### 왜 이 코드가 정답인지

`groupby("category")` 는 같은 카테고리끼리 행을 묶는다. 그 뒤 `count`, `sum`, `mean` 을 적용하면 카테고리별 주문 수, 순매출 합계, 평균 주문금액이 된다. 주문 수가 많은 카테고리와 매출이 큰 카테고리는 다를 수 있다. 이 데이터에서는 기기(`GD`)가 주문 수 1위는 아니지만 단가가 높아 순매출 합계와 평균 주문금액에서 강하게 나타난다.

**예상 핵심값**

| 기준 | 1위 |
|---|---|
| 주문 수 | FD |
| 순매출 합계 | GD |
| 평균 주문금액 | GD |

---

## 문제 12 정답 — 채널과 지역별 매출 집계

In [ ]:
channel_order_count = df["channel"].value_counts()
channel_net_sum = df.groupby("channel")["net_revenue"].sum().sort_values(ascending=False)
channel_ratio = df["channel"].value_counts(normalize=True) * 100
region_net_mean = df.groupby("region")["net_revenue"].mean().sort_values(ascending=False)

print("채널별 주문 수:")
print(channel_order_count)

print("채널별 순매출 합계:")
print(channel_net_sum)

print("채널별 주문 비율(%):")
print(channel_ratio.round(1))

print("지역별 평균 주문금액:")
print(region_net_mean)

print("순매출 합계 1위 채널:", channel_net_sum.idxmax())
print("평균 주문금액 1위 지역:", region_net_mean.idxmax())

### 왜 이 코드가 정답인지

채널 분포는 주문 수와 매출을 함께 봐야 한다. `value_counts()` 는 주문 수 기준 분포를 보여주고, `groupby("channel")["net_revenue"].sum()` 은 매출 기준 분포를 보여준다. `normalize=True` 는 전체 대비 비율을 구하므로 100을 곱하면 퍼센트가 된다. 지역별 평균 주문금액은 지역마다 주문 크기가 다른지 보는 지표다.

**예상 핵심값**

```text
채널 주문 비율: W 43.2%, A 38.0%, S 18.8%
순매출 합계 1위 채널: W
평균 주문금액 1위 지역: C
```

---

## 문제 13 정답 — 월별 매출 흐름

In [ ]:
df["order_date"] = pd.to_datetime(df["date"])
df["order_month"] = df["order_date"].dt.to_period("M")

monthly_order_count = df.groupby("order_month")["order_id"].count()
monthly_net_sum = df.groupby("order_month")["net_revenue"].sum()

print("월별 주문 수:")
print(monthly_order_count)

print("월별 순매출:")
print(monthly_net_sum)

print("순매출 최고 월:", monthly_net_sum.idxmax())
print("순매출 최저 월:", monthly_net_sum.idxmin())

### 왜 이 코드가 정답인지

CSV에서 읽은 날짜는 문자열이므로 날짜 계산을 하려면 `pd.to_datetime` 으로 변환한다. `.dt.to_period("M")` 은 날짜를 월 단위 기간으로 바꾼다. 이렇게 만든 `order_month` 로 groupby 하면 월별 주문 수와 매출 합계를 구할 수 있다. 날짜 분석은 문자열 상태로도 일부 정렬이 되는 것처럼 보일 수 있지만, 날짜형으로 바꾸는 습관이 중요하다.

**예상 핵심값**

| 항목 | 월 |
|---|---|
| 순매출 최고 월 | 2025-01 |
| 순매출 최저 월 | 2025-02 |

---

## 문제 14 정답 — 교차표로 채널×카테고리 보기

In [ ]:
category_channel_count = pd.crosstab(df["category"], df["channel"])
category_channel_ratio = pd.crosstab(df["category"], df["channel"], normalize="index") * 100
category_channel_revenue = df.groupby(["category", "channel"])["net_revenue"].sum().sort_values(ascending=False)

print("카테고리 x 채널 주문 수:")
print(category_channel_count)

print("카테고리별 채널 비율(%):")
print(category_channel_ratio.round(1))

print("카테고리 x 채널 순매출 상위:")
print(category_channel_revenue.head(10))

best_pair = category_channel_revenue.idxmax()
print("순매출 합계 1위 조합:", best_pair)
print("FD의 W 주문 수:", category_channel_count.loc["FD", "W"])

### 왜 이 코드가 정답인지

`pd.crosstab` 은 두 범주형 열의 조합별 개수를 표로 만든다. 행은 카테고리, 열은 채널이다. `normalize="index"` 를 사용하면 각 카테고리 안에서 채널 비율이 얼마나 되는지 볼 수 있다. 매출 합계는 단순 주문 수 교차표가 아니라 `groupby(["category", "channel"])["net_revenue"].sum()` 으로 구한다. `.idxmax()` 는 MultiIndex에서 가장 큰 조합을 튜플로 반환한다.

**예상 핵심값**

```text
FD의 W 주문 수: 100
순매출 합계 1위 조합: ('GD', 'W')
```

---

## 문제 15 정답 — 매출 분석 결론 작성

In [ ]:
total_net = df["net_revenue"].sum()
mean_net = df["net_revenue"].mean()
median_net = df["net_revenue"].median()
best_category_by_revenue = category_net_sum.idxmax()
best_channel_by_revenue = channel_net_sum.idxmax()
best_month_by_revenue = monthly_net_sum.idxmax()

print("총 순매출:", f"{total_net:,.0f}원")
print("평균 주문금액:", f"{mean_net:,.0f}원")
print("중앙 주문금액:", f"{median_net:,.0f}원")
print("순매출 1위 카테고리:", best_category_by_revenue)
print("순매출 1위 채널:", best_channel_by_revenue)
print("순매출 최고 월:", best_month_by_revenue)

### 왜 이 코드가 정답인지

마지막 결론은 앞에서 만든 핵심 지표를 다시 모아 쓰는 문제다. 총 순매출, 평균 주문금액, 중앙 주문금액은 전체 규모를 보여준다. 카테고리, 채널, 월별 1위는 운영 의사결정에 바로 연결되는 항목이다. 결론에 숫자 없이 "좋다", "많다"만 쓰면 데이터 분석 결론이 아니라 감상문이 된다.

**결론 예시**

```text
전체 총 순매출은 37,546,040원이고 평균 주문금액은 37,546원, 중앙 주문금액은 28,050원이다.
평균이 중앙값보다 높으므로 일부 고가 주문이 전체 평균을 끌어올린 것으로 볼 수 있다.
순매출 기준 핵심 카테고리는 GD이고 핵심 채널은 W다.
다음 분석에서는 GD 카테고리의 웹 채널 주문을 더 자세히 보고, 월별 매출 최고/최저 차이의 원인을 확인하겠다.
```

---

## 전체 채점 메모

| 구간 | 문제 | 핵심 개념 | 필수 통과 조건 |
|---|---|---|---|
| 기본 구조 | 1~2 | 로드, shape, dtypes, 결측 | 1,000행 9열과 결측 없음 확인 |
| 선택 | 3~6 | Series/DataFrame, loc/iloc | 열 선택 문법과 행 선택 차이 이해 |
| 필터 | 7~8 | Boolean 조건 | `&` 와 괄호로 복합 조건 작성 |
| 계산 | 9~10 | 계산 열, 정렬 | 순매출 공식과 상위 주문 정렬 |
| 집계 | 11~14 | groupby, 날짜, crosstab | 카테고리/채널/월/교차표 요약 |
| 결론 | 15 | 근거 기반 글쓰기 | 숫자와 1위 항목을 포함한 결론 |

## 학생 답안에서 자주 보는 패턴

| 패턴 | 의미 | 교사 피드백 |
|---|---|---|
| `df['a','b']` | 여러 열 선택 문법 혼동 | `df[['a', 'b']]` 로 고치게 함 |
| `and` 사용 | Series 조건을 스칼라 조건처럼 생각 | `&` 와 괄호를 다시 설명 |
| `gross * discount_rate` 를 순매출로 사용 | 할인액과 순매출 혼동 | 정가 10,000원, 10% 할인 예시로 손계산 |
| `sort_values` 후 원본이 바뀌었다고 생각 | 기본은 새 DataFrame 반환 | 결과를 변수에 담거나 바로 출력하게 함 |
| 날짜를 문자열로 groupby | 이번 데이터에서는 우연히 가능해 보임 | `pd.to_datetime` 후 `.dt` 사용 습관 강조 |
| 결론에 숫자가 없음 | 출력 복기만 함 | 총매출, 평균, 1위 항목 중 2개 이상 포함 |

## 부분 점수 운영 기준

1. 문제 1~2에서 데이터 로드와 구조 확인을 못 하면 이후 계산이 맞아도 재점검하게 한다.
2. 문제 3에서 Series와 DataFrame을 말로 설명하지 못해도, 코드 선택 문법이 맞으면 부분 통과 가능하다.
3. 문제 8에서 `and` 오류가 난 답안은 pandas 필터 핵심이므로 반드시 수정 후 넘어간다.
4. 문제 9의 `net_revenue` 공식이 틀리면 문제 10~15의 매출 결과가 모두 왜곡되므로 재제출로 본다.
5. 문제 15 결론이 코드 출력과 맞지 않으면 코드가 맞아도 결론만 다시 작성하게 한다.

## 수업 중 빠른 확인 질문

- `df["quantity"]` 와 `df[["quantity"]]` 중 Series는 어느 쪽인가?
- `df.loc[:5]` 와 `df.iloc[:5]` 중 6행이 나오는 것은 어느 쪽인가?
- 할인율 20% 주문의 순매출은 정가 매출의 몇 퍼센트인가?
- `value_counts(normalize=True)` 에 100을 곱하면 어떤 단위가 되는가?
- `groupby("category")["net_revenue"].sum()` 과 `value_counts()` 는 각각 어떤 질문에 답하는가?

학생이 이 질문들에 답하면 03강의 핵심 개념은 대체로 통과한 것이다. pandas 수업 초반에는 화려한 코드를 쓰는 것보다 "어떤 열을 선택했고, 어떤 행을 남겼고, 어떤 기준으로 묶었는지"를 또렷하게 말하는 능력을 우선 본다.

## 문제별 지도 질문

| 문제 | 학생에게 던질 질문 | 확인할 답 |
|---:|---|---|
| 1 | `df.shape` 의 두 숫자는 각각 무엇인가요? | 행 수와 열 수 |
| 2 | `info()` 에서 결측 여부는 어디를 보면 되나요? | Non-Null Count |
| 3 | 열 하나와 여러 열을 선택할 때 대괄호가 어떻게 다른가요? | 문자열 하나 vs 열 이름 리스트 |
| 4 | 평균과 중앙값이 다른 이유는 무엇인가요? | 큰 값의 영향 여부 |
| 5 | `value_counts()` 는 어떤 열에 적합한가요? | 범주형 코드 열 |
| 6 | `loc` 와 `iloc` 의 기준은 각각 무엇인가요? | 라벨과 위치 |
| 7 | 할인율 10% 이상을 코드로 쓰면 어떤 비교 연산자인가요? | `>= 0.10` |
| 8 | pandas 조건에서 `and` 대신 무엇을 쓰나요? | `&` 와 괄호 |
| 9 | 할인 금액과 순매출은 어떻게 다른가요? | 할인액은 빠지는 금액, 순매출은 실제 지불액 |
| 10 | 정렬 후 원본 `df` 의 행 순서가 자동으로 바뀌나요? | 기본은 바뀌지 않음 |
| 11 | `groupby` 는 어떤 질문에 답하나요? | 그룹별 합계/평균/개수 |
| 12 | 주문 비율은 어떤 옵션으로 구하나요? | `normalize=True` |
| 13 | `.dt` 접근자는 언제 사용할 수 있나요? | 날짜형 Series에서 사용 |
| 14 | 교차표에서 특정 칸은 어떻게 꺼내나요? | `.loc[행라벨, 열라벨]` |
| 15 | 좋은 결론과 나쁜 결론의 차이는 무엇인가요? | 출력값 근거 포함 여부 |

## 보충 설명 포인트

- pandas 첫 수업에서는 암기보다 표 감각이 중요하다. 학생에게 "행을 줄였는지, 열을 줄였는지, 새 열을 만들었는지"를 계속 말하게 한다.
- `Series` 와 `DataFrame` 타입 차이는 이후 에러 메시지 이해와 직접 연결된다. 같은 `quantity` 열이라도 `df["quantity"]` 와 `df[["quantity"]]` 의 출력 모양을 나란히 보여주면 효과적이다.
- 조건 필터는 괄호가 핵심이다. 학생이 연산자 우선순위를 아직 몰라도 `(조건1) & (조건2)` 형태를 습관으로 만들면 실수를 크게 줄일 수 있다.
- 매출 공식은 실제 손계산 예시로 확인한다. 수량 2개, 단가 10,000원, 할인율 10%라면 정가 매출 20,000원, 할인액 2,000원, 순매출 18,000원이다.
- `groupby` 는 `value_counts()` 의 확장으로 설명하면 자연스럽다. 개수만 세던 것을 합계, 평균, 중앙값 같은 여러 집계로 확장한다고 연결한다.
- 날짜 분석은 문자열 그대로도 일부 결과가 나와서 학생이 착각하기 쉽다. 날짜형으로 바꿔야 월, 요일, 기간 계산이 안정적으로 가능하다는 점을 강조한다.